# <span style ="color:#0b486b">FIT5149 Applied Data Analysis</span>

## <span style ="color:#0b486b">Week 6 Applied Class: CNN vs a Mini Vision Transformer</span>

In Week 5 you built the components of a transformer and used a decoder-only transformer for next-token prediction. This week you will reuse **exactly those components** to build a tiny Vision Transformer (ViT), and compare it with a convolutional neural network on the same image classification task.

The key idea: **a patch is to an image what a subword is to a sentence.** Splitting an image into patches plays the same role as tokenisation in NLP; after that step, the transformer machinery is identical. In fact, the only differences between this week's vision model and last week's language model are the absence of the causal mask and the classification head.

By the end of this class you should be able to:

* compute the output shape and parameter count of convolution and pooling layers;
* explain patchification as tokenisation for images;
* assemble a Vision Transformer from the Week 5 building blocks;
* compare a CNN and a ViT in terms of parameters, training time and accuracy, and compare two optimisers for training a transformer;
* explain the outcome in terms of inductive bias, and why ViTs win only at scale.

**Solutions version.** All tasks include model solutions.

*Acknowledgement: the dataset and loading pipeline are adapted from the Week 8 applied class on CNNs.*


### <span style ="color:#0b486b">I. Preparation</span>

#### <span style ="color:#0b486b">I.1 Setup</span>


In [ ]:
import os
import math
import time
import random

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on", device)
torch.manual_seed(1234)


#### <span style ="color:#0b486b">I.2 Dataset</span>

We use the 20-class image dataset (`Dataset.zip`), organised in class folders for `ImageFolder`. All images are resized to $64 \times 64$ and normalised to $[-1, 1]$; the training set additionally uses light augmentation.

One improvement over the earlier pipeline: augmentation is applied **only to the training split**. We load the folder twice, once with the training transform and once with the evaluation transform, and index both with the same shuffled indices, so the validation and test sets are evaluated on clean images.


In [ ]:
!unzip -q -n Dataset.zip
# Alternatively, unzip Dataset.zip using dedicated archive extraction software.

In [ ]:
# Dataset directory path
data_dir = "./Dataset"

class_names = ['bird', 'bottle', 'bread', 'butterfly', 'cake', 'cat', 'chicken', 'cow', 'dog', 'duck',
               'elephant', 'fish', 'handgun', 'horse', 'lion', 'lipstick', 'seal', 'snake', 'spider', 'vase']
num_classes = len(class_names)

image_size = 64

# Training transform (with light augmentation)
transform_train = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Evaluation transform (no augmentation)
transform_test = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load the folder twice: same files, different transforms
train_view = datasets.ImageFolder(data_dir, transform=transform_train)
eval_view = datasets.ImageFolder(data_dir, transform=transform_test)
print(f"Loaded {len(train_view)} images in {len(train_view.classes)} classes")

# CLASSROOM MODE: use a subset for faster CPU training. Increase if your machine allows.
max_samples = 2000
random.seed(2025)
all_indices = list(range(len(train_view)))
random.shuffle(all_indices)
subset_indices = all_indices[:min(max_samples, len(all_indices))]

# Split the subset indices: 70% train, 15% validation, 15% test
n = len(subset_indices)
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_idx = subset_indices[:n_train]
val_idx = subset_indices[n_train:n_train + n_val]
test_idx = subset_indices[n_train + n_val:]

train_dataset = Subset(train_view, train_idx)   # augmented view
val_dataset = Subset(eval_view, val_idx)        # clean view
test_dataset = Subset(eval_view, test_idx)      # clean view

print(f"Training: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# Visualise a batch of training images
def show_images(images, labels, n=8):
    plt.figure(figsize=(12, 2))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        img = images[i] / 2 + 0.5                      # undo the normalisation
        plt.imshow(np.transpose(img.numpy(), (1, 2, 0)))
        plt.title(class_names[labels[i]], fontsize=8)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)
show_images(images, labels)


### <span style ="color:#0b486b">II. The Week 5 Building Blocks</span>

The cell below contains the transformer components exactly as you implemented them in Week 5, now with comments explaining what each piece does: scaled dot-product attention, multi-head self-attention, the sinusoidal positional encoding and the transformer block. Nothing here is new; read through the comments, run the cell and move on.

Note the `mask` argument: last week the language model passed a causal mask so each token could only see the past. This week we will pass `mask=None`, because every patch of an image is allowed to attend to every other patch. That single argument is the difference between a decoder-style and an encoder-style transformer.


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """The core attention operation: Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V.

    Q: (..., n_q, d_k) queries, K: (..., n_k, d_k) keys, V: (..., n_k, d_v) values.
    Each query produces a weighted average of the values, where the weights are a softmax
    over the similarity (dot product) between the query and every key.
    mask: optional (n_q, n_k) tensor of 0s and 1s; positions with 0 receive zero weight.
    Returns the output (..., n_q, d_v) and the attention weights (..., n_q, n_k).
    """
    d_k = Q.size(-1)
    # Similarity of every query with every key, scaled by sqrt(d_k) so that the dot
    # products do not grow with the dimension (which would saturate the softmax).
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)     # (..., n_q, n_k)
    # Masked positions are set to -inf BEFORE the softmax, so they get exactly zero weight.
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    # Softmax over the key dimension: each row is a probability distribution over the keys.
    weights = torch.softmax(scores, dim=-1)
    # Each output is the weighted sum of the values.
    return torch.matmul(weights, V), weights


class MultiHeadSelfAttention(nn.Module):
    """Self-attention with several heads computed in parallel.

    Q, K and V are all linear projections of the same input x (hence "self"-attention).
    The d_model-dimensional space is split into num_heads sub-spaces of size d_head, each
    head attends independently, and the head outputs are concatenated and mixed by out_proj.
    Different heads can learn different relationships (e.g. neighbouring patches, same colour).
    """

    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        # One linear layer produces Q, K and V at once (3 * d_model outputs) for efficiency.
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        # Mixes the concatenated heads back into a d_model-dimensional output.
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        # x: (batch, seq_len, d_model)
        B, T, D = x.shape
        # Step 1: project and split into Q, K, V, each (B, T, D).
        Q, K, V = self.qkv_proj(x).chunk(3, dim=-1)
        # Step 2: reshape to (B, num_heads, T, d_head) so every head attends in parallel.
        Q = Q.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        # Step 3: attention per head; a (T, T) mask broadcasts over batch and heads.
        out, _ = scaled_dot_product_attention(Q, K, V, mask=mask)      # (B, heads, T, d_head)
        # Step 4: concatenate the heads back to (B, T, D) and mix them.
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)


def sinusoidal_positional_encoding(seq_len, d_model):
    """Fixed (not learned) position vectors, one per position, of dimension d_model.

    Even dimensions hold sin(pos / 10000^(2i/d_model)), odd dimensions the matching cos,
    so each dimension oscillates at its own frequency, from fast (low i) to slow (high i).
    Adding this matrix to the token embeddings gives the model information about order,
    which attention alone does not have. Returns a (seq_len, d_model) tensor.
    """
    position = torch.arange(seq_len).unsqueeze(1)                        # (seq_len, 1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(seq_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)                         # even dimensions
    pe[:, 1::2] = torch.cos(position * div_term)                         # odd dimensions
    return pe


class TransformerBlock(nn.Module):
    """One transformer layer: self-attention, then a position-wise feed-forward network.

    Each sub-layer is wrapped in a residual connection (x + sublayer(x)) and preceded by
    layer normalisation ("pre-LN"). The residual connections let gradients flow directly
    through deep stacks (avoiding vanishing gradients) and the normalisation keeps the
    activations well scaled. Attention mixes information ACROSS positions; the feed-forward
    network transforms each position independently.
    """

    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        # Two-layer MLP applied to every position separately; d_ff is usually 4 * d_model.
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # x: (batch, seq_len, d_model); the output has the same shape.
        x = x + self.attn(self.ln1(x), mask=mask)   # attention sub-layer + residual connection
        x = x + self.ff(self.ln2(x))                # feed-forward sub-layer + residual connection
        return x

print("Week 5 components loaded.")


### <span style ="color:#0b486b">III. Shared Training and Evaluation Code</span>

Both models will be trained with the same functions, so that the comparison is fair: same data, same loss, same number of epochs. Only the optimiser differs, and that difference will turn out to matter.


In [ ]:
criterion = nn.CrossEntropyLoss()   # softmax + negative log-likelihood, for multi-class classification

def run_epoch(model, loader, optimizer=None):
    """One full pass over `loader`. Trains if an optimizer is given, otherwise only evaluates.

    - model.train() / model.eval() switches the behaviour of BatchNorm and dropout layers:
      in eval mode BatchNorm uses its running statistics instead of the batch statistics.
    - torch.set_grad_enabled(training) turns gradient tracking off during evaluation,
      which saves memory and time.
    Returns the mean loss and the accuracy over the whole loader.
    """
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)                     # forward pass: logits (batch, num_classes)
            loss = criterion(outputs, labels)
            if training:
                optimizer.zero_grad()                   # clear gradients from the previous step
                loss.backward()                         # backward pass: compute gradients
                optimizer.step()                        # update the parameters
            total_loss += loss.item() * labels.size(0)  # loss is a batch mean, so re-weight by batch size
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

def train_model(model, optimizer, num_epochs):
    """Trains for `num_epochs`, printing training and validation loss/accuracy after each epoch.

    Returns the wall-clock training time in seconds, which we use in the comparison table.
    """
    start = time.time()
    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer)   # one training pass
        val_loss, val_acc = run_epoch(model, val_loader)                    # one evaluation pass
        print(f"Epoch {epoch:2d}/{num_epochs} | "
              f"train loss {train_loss:.3f} acc {train_acc:.3f} | "
              f"val loss {val_loss:.3f} acc {val_acc:.3f}")
    elapsed = time.time() - start
    print(f"Training took {elapsed:.1f}s")
    return elapsed

def count_parameters(model):
    """Number of trainable parameters (weights and biases) in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


### <span style ="color:#0b486b">IV. The CNN Baseline</span>

#### <span style ="color:#0b486b">IV.1 Convolution and pooling layers in PyTorch</span>

A CNN is built from two kinds of layers. Their parameters, and the shapes they produce, are worth understanding precisely before you build anything.

**`nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)`**

* `in_channels`: the depth of the input volume (3 for an RGB image, or the number of filters of the previous layer). Every filter spans the **full** input depth.
* `out_channels`: the number of filters; each filter produces one output feature map, so this is also the depth of the output.
* `kernel_size`: the height and width of the sliding window, e.g. `5` for a $5 \times 5$ filter.
* `stride`: how far the window moves between positions; stride $2$ roughly halves the output size.
* `padding`: rows and columns of zeros added around the border. With stride $1$, `padding = (kernel_size - 1) / 2` keeps the spatial size unchanged (“same” padding).

The weight tensor has shape `(out_channels, in_channels, kernel_size, kernel_size)`, plus one bias per filter, so the number of parameters is
$$\text{params} = \text{out\_channels} \times (\text{in\_channels} \times k \times k) + \text{out\_channels},$$
which does **not** depend on the image size: the same filter is reused at every position (weight sharing).

**`nn.MaxPool2d(kernel_size, stride=None, padding=0)`**

* `kernel_size`: the window over which the maximum is taken.
* `stride`: defaults to `kernel_size`, so `MaxPool2d(2, 2)` halves the height and width.
* `padding`: as for convolution.

Pooling has **no** learnable parameters: it summarises each window by its maximum, which gives some robustness to small translations.

**Output size.** Both layers slide a window of size $k$ with stride $s$ over an input of size $H_{in}$ padded by $p$ on each side, so both follow the same formula:
$$H_{out} = \left\lfloor \frac{H_{in} + 2p - k}{s} \right\rfloor + 1 \qquad (\text{and identically for the width}).$$

Worked example: a $64 \times 64$ input through `Conv2d(k=5, s=1, p=2)` gives $\lfloor (64 + 4 - 5)/1 \rfloor + 1 = 64$, and then `MaxPool2d(2, 2)` gives $\lfloor (64 - 2)/2 \rfloor + 1 = 32$.

**<span style="color:red">Task 1:</span>** Complete the two functions below.

* **(a)** `conv_out_size(h_in, kernel_size, stride, padding)`: the output size given by the formula above. It is then used to predict the output shape of a chain of layers **before** the layers are run, and the self-checks compare your predictions against what PyTorch actually produces.
* **(b)** `conv_num_parameters(in_channels, out_channels, kernel_size)`: the number of learnable parameters of a `Conv2d` layer, from the parameter formula above. Compute it from the arguments rather than counting the entries of the layer object. The self-checks compare your count against `count_parameters` for two different convolutions, and confirm that a pooling layer has none.


In [ ]:
def conv_out_size(h_in, kernel_size, stride, padding):
    # (a) H_out = floor((H_in + 2p - k) / s) + 1, the same formula for convolution and pooling
    return (h_in + 2 * padding - kernel_size) // stride + 1

def conv_num_parameters(in_channels, out_channels, kernel_size):
    # (b) each of the out_channels filters spans the FULL input depth, so it holds
    #     in_channels * k * k weights, and each filter has one bias.
    #     The count does not involve the input height or width: the same filters are
    #     reused at every spatial position (weight sharing).
    return out_channels * (in_channels * kernel_size * kernel_size) + out_channels

# ---------------- self-checks ----------------
x = torch.randn(2, 3, 64, 64)                   # a batch of 2 RGB images

layers = [
    ("Conv2d k=5 s=1 p=2", nn.Conv2d(3, 16, kernel_size=5, stride=1, padding=2), (5, 1, 2)),
    ("MaxPool2d k=2 s=2", nn.MaxPool2d(kernel_size=2, stride=2),                  (2, 2, 0)),
    ("Conv2d k=3 s=2 p=1", nn.Conv2d(16, 8, kernel_size=3, stride=2, padding=1), (3, 2, 1)),
    ("MaxPool2d k=3 s=2 p=1", nn.MaxPool2d(kernel_size=3, stride=2, padding=1),   (3, 2, 1)),
]

# (a) predicted vs actual output shapes: 64 -> 64 -> 32 -> 16 -> 8
h, out = 64, x
for name, layer, (k, s, p) in layers:
    h = conv_out_size(h, k, s, p)                # your prediction
    out = layer(out)                             # what PyTorch actually produces
    assert out.shape[-1] == h and out.shape[-2] == h, f"{name}: predicted {h}, got {tuple(out.shape)}"
    print(f"{name:<22} -> {tuple(out.shape)}   parameters: {count_parameters(layer)}")

# (b) predicted vs actual parameter counts
conv1, conv2, pool1 = layers[0][1], layers[2][1], layers[1][1]
assert conv1.weight.shape == (16, 3, 5, 5)                        # (out, in, k, k), plus one bias each
assert conv_num_parameters(3, 16, 5) == count_parameters(conv1)   # 16 * (3*5*5) + 16 = 1216
assert conv_num_parameters(16, 8, 3) == count_parameters(conv2)   # 8 * (16*3*3) + 8  = 1160
assert count_parameters(pool1) == 0                               # pooling has no learnable parameters
print(f"Conv2d(3, 16, k=5):    {conv_num_parameters(3, 16, 5)} parameters")
print(f"Conv2d(16, 8, k=3):    {conv_num_parameters(16, 8, 3)} parameters")

# The count is independent of the input size: the same filters are reused at every position.
assert conv_num_parameters(3, 16, 5) == count_parameters(nn.Conv2d(3, 16, kernel_size=5, padding=2))
print("All checks passed!")


#### <span style ="color:#0b486b">IV.2 The CNN</span>

We adapt the compact CNN from the Week 5 lab to this dataset. The architecture is unchanged, two convolution blocks followed by a fully connected layer, but the input is now $3 \times 64 \times 64$ instead of $1 \times 28 \times 28$, and there are $20$ classes.

**<span style="color:red">Task 2:</span>** Complete the CNN below. You need to work out the input size of the final fully connected layer. Trace the shapes with the formula from Task 1: each `Conv2d` uses kernel $5$, stride $1$, padding $2$, so it preserves the spatial size, and each `MaxPool2d` halves it. Start from $64 \times 64$ and count.


In [ ]:
class ConvNet(nn.Module):

    def __init__(self, num_classes=20):
        super().__init__()
        # input shape: (None, 3, 64, 64)
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, stride=1, padding=2),   # (None, 16, 64, 64)
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))                  # (None, 16, 32, 32)
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, stride=1, padding=2),  # (None, 32, 32, 32)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))                  # (None, 32, 16, 16)
        # Each convolution preserves the spatial size (padding = (kernel-1)/2, stride 1),
        # each pooling halves it: 64 -> 32 -> 16. Flattened size = 32 * 16 * 16 = 8192.
        self.fc = nn.Linear(32 * 16 * 16, num_classes)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        return self.fc(out)

# ---------------- self-checks ----------------
cnn_model = ConvNet(num_classes).to(device)
dummy = torch.randn(2, 3, image_size, image_size, device=device)
assert cnn_model(dummy).shape == (2, num_classes)
print(f"CNN parameters: {count_parameters(cnn_model):,}")
print("All checks passed!")


In [ ]:
num_epochs = 8

optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
cnn_time = train_model(cnn_model, optimizer, num_epochs)
_, cnn_test_acc = run_epoch(cnn_model, test_loader)
print(f"CNN test accuracy: {cnn_test_acc:.3f}")


### <span style ="color:#0b486b">V. Demo: Patchification, Tokenisation for Images</span>

A transformer consumes a **sequence of vectors**. Last week the sequence came from a tokeniser: BPE split “unbelievable” into `un | believ | able`, and each subword became a vector via an embedding table. An image is not a sequence, so we manufacture one: cut the image into a grid of non-overlapping patches and flatten each patch into a vector.

With $64 \times 64$ images and $8 \times 8$ patches we get an $8 \times 8$ grid, i.e. a sequence of $64$ “tokens”, each a vector of $8 \times 8 \times 3 = 192$ pixel values. A patch is to an image what a subword is to a sentence.

The implementation below uses `Tensor.unfold(dim, size, step)`, which extracts sliding windows along one dimension; applying it to the height and then the width dimension with `size = step = patch_size` gives non-overlapping patches. The windows are then rearranged so that the patch grid becomes one sequence dimension and each patch is flattened into a vector. The self-checks verify the shape and that every pixel appears exactly once, and the plot shows one image as its sequence of 64 patch tokens.


In [ ]:
def patchify(images, patch_size):
    # images: (B, C, H, W) -> returns (B, num_patches, C * patch_size * patch_size)
    B, C, H, W = images.shape
    P = patch_size
    # Step 1: extract non-overlapping P x P windows -> (B, C, H/P, W/P, P, P)
    patches = images.unfold(2, P, P).unfold(3, P, P)
    # Step 2: move the grid dimensions before the channel dimension
    patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()   # (B, H/P, W/P, C, P, P)
    # Step 3: flatten the grid into a sequence, and each patch into a vector
    return patches.view(B, (H // P) * (W // P), C * P * P)

# ---------------- self-checks ----------------
patch_size = 8
x = images[:2]                                     # a small batch from the loader
patches = patchify(x, patch_size)
num_patches = (image_size // patch_size) ** 2
patch_dim = 3 * patch_size * patch_size
assert patches.shape == (2, num_patches, patch_dim), patches.shape
assert torch.allclose(patches.sum(), x.sum(), atol=1e-4)     # every pixel appears exactly once
print(f"{num_patches} patches per image, each of dimension {patch_dim}")
print("All checks passed!")

# Visualise the patch grid of one image
plt.figure(figsize=(4, 4))
for i in range(num_patches):
    plt.subplot(8, 8, i + 1)
    patch = patches[0, i].view(3, patch_size, patch_size) / 2 + 0.5
    plt.imshow(np.transpose(patch.numpy(), (1, 2, 0)))
    plt.axis('off')
plt.suptitle("One image as a sequence of 64 patch tokens")
plt.show()


### <span style ="color:#0b486b">VI. Vision Transformers</span>

The **Vision Transformer (ViT)** was introduced by Dosovitskiy et al. (2021) in the paper *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale* [1]. Its central claim is that the transformer encoder from NLP can be applied to images essentially unchanged, provided the image is first turned into a sequence of tokens. The recipe:

1. **Patchify**: split the image into fixed-size patches (the paper uses $16 \times 16$ pixels on $224 \times 224$ images, giving $196$ tokens; we use $8 \times 8$ on $64 \times 64$, giving $64$).
2. **Embed**: flatten each patch and map it to $d_{\text{model}}$ dimensions with a single linear layer. This replaces the `nn.Embedding` lookup of the language model; the projection is the same for every patch, exactly as an embedding table is shared across positions.
3. **Add position information**: attention has no notion of where a patch came from, so a position embedding is added to every token. The paper learns these; we use the sinusoidal encoding from Week 5.
4. **Encode**: run a stack of standard transformer blocks **without a causal mask**. Every patch attends to every other patch, so the model can relate distant parts of the image from the very first layer, something a convolution can only do after many layers.
5. **Classify**: the paper prepends a learnable `[CLS]` token and reads the prediction from its final state; we use the simpler alternative of averaging the final token vectors and applying a linear head.

The paper's main empirical finding is the reason this class exists: with **little** training data, ViTs are worse than CNNs of similar size, because a transformer must learn from data the locality and translation structure that convolution builds in; with **very large** pre-training data (tens to hundreds of millions of images) the ViT overtakes the best CNNs. You are about to reproduce the first half of that finding.

[1] Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., Dehghani, M., Minderer, M., Heigold, G., Gelly, S., Uszkoreit, J. and Houlsby, N. (2021). An image is worth 16x16 words: Transformers for image recognition at scale. *International Conference on Learning Representations (ICLR)*. https://arxiv.org/abs/2010.11929

#### <span style ="color:#0b486b">VI.1 Using a transformer block on patch tokens</span>

Before assembling the full model, let us apply one `TransformerBlock` directly to the patch tokens and verify two properties that explain the design above.

**<span style="color:red">Task 3:</span>** Complete the cell below.

* (a) Embed the patches of two images with a linear layer and pass them through a `TransformerBlock` with `mask=None`. Check that the block preserves the sequence shape `(B, 64, d_model)`.
* (b) Shuffle the 64 patches of each image with a random permutation `perm`, pass them through the same block, and compare the **mean-pooled** outputs of the original and the shuffled sequence. Without positional encodings they are identical: the block is permutation-equivariant and averaging is permutation-invariant, so such a model cannot tell an image from a scrambled version of it.
* (c) Add the sinusoidal positional encoding to the tokens before the block and repeat (b). Now the pooled outputs differ: the positional encoding is what lets the model know where each patch was.


In [ ]:
torch.manual_seed(0)
d_model, num_heads = 64, 4
block = TransformerBlock(d_model, num_heads, 4 * d_model)
embed = nn.Linear(patch_dim, d_model)
tokens = embed(patchify(images[:2], patch_size))          # (2, 64, d_model)

# (a) a transformer block maps a sequence to a sequence of the same shape
out = block(tokens, mask=None)
assert out.shape == tokens.shape

# (b) shuffle the patches: the block is permutation-equivariant (the output rows are
#     shuffled in the same way) and mean-pooling is permutation-invariant, so the pooled
#     outputs coincide. A ViT without positional encodings cannot see the arrangement of patches.
perm = torch.randperm(num_patches)
pooled = block(tokens, mask=None).mean(dim=1)
pooled_perm = block(tokens[:, perm], mask=None).mean(dim=1)
same_without_pe = torch.allclose(pooled, pooled_perm, atol=1e-5)
print("Without PE, scrambled image gives the same pooled output:", same_without_pe)

# (c) with positional encodings added BEFORE the block, position i always receives the same
#     encoding whichever patch sits there, so shuffling the patches changes the input and the output.
pe = sinusoidal_positional_encoding(num_patches, d_model)
pooled_pe = block(tokens + pe, mask=None).mean(dim=1)
pooled_pe_perm = block(tokens[:, perm] + pe, mask=None).mean(dim=1)
same_with_pe = torch.allclose(pooled_pe, pooled_pe_perm, atol=1e-5)
print("With PE, scrambled image gives the same pooled output:   ", same_with_pe)

assert same_without_pe and not same_with_pe
print("All checks passed!")


#### <span style ="color:#0b486b">VI.2 Assembling the Mini Vision Transformer</span>

The model is the Week 5 language model with three substitutions:

| | `MiniTransformerLM` (Week 5) | `MiniViT` (this week) |
|---|---|---|
| Tokens | subword ids | image patches |
| Embedding | `nn.Embedding` lookup | `nn.Linear(192, 64)` projection |
| Mask | causal (`torch.tril`) | `None`: every patch sees every patch |
| Head | next-token logits at every position | mean-pool, then one 20-class prediction |

Everything in the middle, positional encodings and the stack of transformer blocks, is identical.

**<span style="color:red">Task 4:</span>** Complete the `forward` method of `MiniViT`: patchify, project and add positional encodings, run the transformer blocks with `mask=None`, apply the final layer norm, average over the patch dimension, and classify. Task 3 contains every operation you need.


In [ ]:
class MiniViT(nn.Module):

    def __init__(self, image_size=64, patch_size=8, in_channels=3, num_classes=20,
                 d_model=64, num_heads=4, num_layers=2):
        super().__init__()
        self.patch_size = patch_size
        num_patches = (image_size // patch_size) ** 2
        patch_dim = in_channels * patch_size * patch_size
        self.patch_embed = nn.Linear(patch_dim, d_model)
        self.register_buffer("pos_enc", sinusoidal_positional_encoding(num_patches, d_model))
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, num_heads, 4 * d_model) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        # x: (B, 3, 64, 64)
        p = patchify(x, self.patch_size)          # (B, 64, 192): the "token" sequence
        h = self.patch_embed(p) + self.pos_enc    # (B, 64, d_model)
        for block in self.blocks:
            h = block(h, mask=None)               # no causal mask: an encoder, not a decoder
        h = self.ln_f(h).mean(dim=1)              # (B, d_model): average over the patches
        return self.head(h)                       # (B, num_classes)

# ---------------- self-checks ----------------
vit_model = MiniViT(image_size=image_size, num_classes=num_classes).to(device)
dummy = torch.randn(2, 3, image_size, image_size, device=device)
assert vit_model(dummy).shape == (2, num_classes)
print(f"ViT parameters: {count_parameters(vit_model):,}")
print("All checks passed!")


### <span style ="color:#0b486b">VII. Training the ViT: Comparing Two Optimisers</span>

The CNN trained without fuss using Adam with learning rate $0.001$. Transformers are considerably more sensitive to the optimiser and learning rate. Two reasons: their parameters (embeddings, attention projections, layer norms, feed-forward weights) receive gradients of very different scales, and their loss surface is sharper, so a fixed step size that suits one group of parameters is too large for another. Adaptive optimisers such as Adam and AdamW scale each parameter's step by a running estimate of its gradient magnitude, which is why they are the standard choice for transformers (usually with a small learning rate and, in large runs, a warm-up schedule). AdamW differs from Adam in applying weight decay directly to the weights rather than through the gradient, which regularises transformers more reliably.

**<span style="color:red">Task 5:</span>** Train two copies of the ViT from the same initialisation, with the same data and number of epochs, and compare:

* **Setting A**: `torch.optim.AdamW`, learning rate `3e-4`.
* **Setting B**: plain `torch.optim.SGD`, learning rate `0.1` (the setting that trained the XOR network in Week 5).

Watch the training loss in each run. Random guessing over $20$ classes gives a cross-entropy of $-\ln(1/20) \approx 3.0$; a run whose loss stays near that value is not learning. Setting A is kept as “the ViT” for the comparison with the CNN in the next section.


In [ ]:
def make_vit(seed=0):
    torch.manual_seed(seed)                                   # same initial weights for both runs
    return MiniViT(image_size=image_size, num_classes=num_classes).to(device)

# ---- Setting A: AdamW, lr = 3e-4
vit_adamw = make_vit()
optimizer = torch.optim.AdamW(vit_adamw.parameters(), lr=3e-4)
adamw_time = train_model(vit_adamw, optimizer, num_epochs)
_, adamw_test_acc = run_epoch(vit_adamw, test_loader)

# ---- Setting B: SGD, lr = 0.1
vit_sgd = make_vit()
optimizer = torch.optim.SGD(vit_sgd.parameters(), lr=0.1)
sgd_time = train_model(vit_sgd, optimizer, num_epochs)
_, sgd_test_acc = run_epoch(vit_sgd, test_loader)

print(f"\n{'optimiser':<14}{'train time (s)':>16}{'test accuracy':>16}")
print("-" * 46)
print(f"{'AdamW 3e-4':<14}{adamw_time:>16.1f}{adamw_test_acc:>16.3f}")
print(f"{'SGD 0.1':<14}{sgd_time:>16.1f}{sgd_test_acc:>16.3f}")

# keep the AdamW model as "the ViT" for the comparison with the CNN
vit_model, vit_time, vit_test_acc = vit_adamw, adamw_time, adamw_test_acc


Typically the AdamW run learns steadily while the SGD run stalls near chance level or oscillates; occasionally SGD makes slow progress, but it is far behind after the same number of epochs. Note that the two optimisers cost the same time per epoch: what differs is how far each step moves the model in a useful direction, not how expensive the step is.


### <span style ="color:#0b486b">VIII. The Comparison</span>

Both models saw exactly the same data for the same number of epochs. Let us put the numbers side by side.


In [ ]:
print(f"{'model':<12}{'parameters':>14}{'train time (s)':>16}{'test accuracy':>16}")
print("-" * 58)
print(f"{'CNN':<12}{count_parameters(cnn_model):>14,}{cnn_time:>16.1f}{cnn_test_acc:>16.3f}")
print(f"{'Mini ViT':<12}{count_parameters(vit_model):>14,}{vit_time:>16.1f}{vit_test_acc:>16.3f}")


With only a couple of thousand images, both accuracies are modest, and that is fine; what matters is the comparison. The two models have a comparable number of parameters, yet the CNN typically reaches a clearly higher accuracy: convolution builds locality and translation equivariance into the architecture, while attention must learn them from data, which is exactly the small-data regime in which Dosovitskiy et al. report ViTs falling behind.

**<span style="color:red">Task 6 (discussion):</span>** Discuss the following with your neighbour and write short answers.

1. Suppose you change the patch size from $8 \times 8$ to $4 \times 4$. How many patch tokens does one image produce, and what happens to the cost of self-attention?
2. The `MiniViT` passes `mask=None` to the transformer blocks, while last week's `MiniTransformerLM` could not be trained without a causal mask. Explain the difference.


**Sample answers.**

**1.** With $4 \times 4$ patches, a $64 \times 64$ image gives a $16 \times 16$ grid, i.e. $256$ tokens instead of $64$. Self-attention computes an attention weight for every pair of tokens, so its cost is quadratic in the sequence length: $4\times$ more tokens means roughly $16\times$ more attention computation (and $16\times$ larger attention matrices). Finer patches capture finer detail but pay quadratically for it.

**2.** The language model is trained to predict the *next* token, and the target at position $t$ sits inside the same input sequence at position $t+1$; without the causal mask the model could simply copy it, so masking is what forces genuine prediction. The ViT predicts a single class label that is not part of the input sequence at all, so there is nothing to leak: every patch may freely attend to every other patch, and bidirectional context strictly helps. The one-line difference, `mask=None` versus a `torch.tril` mask, is precisely the difference between an encoder and a decoder.
